# Formal H=800 naive CRL (300k) -- rockfall v2.1 local-detour, p_active=0.30 (Colab GPU)\n
Faithful offline CRL on the **90/0/10 local-detour** primary pilot (270 sighted /
0 blind / 30 center) at **v2.1 severity 0.80/0.15/0.05**. Learner sees only
obs/act. Eval runs in `offline_ant_umaze_rockfall` under the v2.1 severity
(cfg.rockfall_severity). 300k-step run to PIN THE PLATEAU (40k was undertrained: the
curve was still climbing at 0.60). Eval every 10k; stop early / use best.pkl if
it plateaus. If still climbing at 300k, set RESUME=True and rerun. Dataset (27MB) ships with the
repo at the pinned commit. Behavioural metrics run on the WORKSTATION after
download: `python scripts/diagnose_naive_rockfall.py --v2 --ckpt <dir>/best.pkl`.


## 1. Configuration -- single source of truth

In [ ]:
# 1. Configuration. Everything tunable lives here.
import os

SEED        = 0
MAX_UPDATES = 300_000         # pin the plateau (was 40k, still climbing)
RESUME      = False           # set True after a Colab disconnect (restores latest.pkl)
REQUIRE_GPU = True
P_ACTIVE    = 0.3          # mask density for THIS sweep condition
HORIZON     = 800         # H=800 experiment (H=700 default is untouched)

# --- repo (pinned to the rockfall dataset commit) ------------------------------
REPO     = 'contrastive_rl'
REPO_URL = 'https://github.com/tingrui-huang/contrastive_rl.git'
BRANCH   = 'main'
COMMIT   = '197fac9'          # H=800 experiment frozen commit

# --- dataset: ships WITH the repo at this commit --------------------------------
ENV_NAME             = 'offline_ant_umaze_rockfall'
DATASET_REPO_RELPATH = 'artifacts/rockfall_v2_p30_h800/pilot/antmaze_rockfall_v2_p30_h800_pilot.npz'
DATASET_SHA256       = '45c4db1dff63f0c3b263d91fc38e33449cf98484173b510b93d37047c263df33'

# --- run dirs --------------------------------------------------------------------
RUN_ID            = f'naive_rockfall_v2_p30_h800_s{SEED}_{MAX_UPDATES//1000}k'
LOCAL_RUN_DIR     = f'/content/runs/{RUN_ID}'
RUN_DRIVE_DIR     = f'/content/drive/MyDrive/contrastive_rl_runs/{RUN_ID}'
LOCAL_DATASET_PATH = f'/content/{REPO}/{DATASET_REPO_RELPATH}'
print(RUN_ID)


## 2. Mount Google Drive; create dirs

In [ ]:
# 2. Mount Drive; make local scratch + Drive run trees.
# (The rockfall dataset ships WITH the repo -- Drive is only for checkpoint
#  mirroring, so there is no dataset dir to create here.)
from google.colab import drive
drive.mount('/content/drive')
for base in (LOCAL_RUN_DIR, RUN_DRIVE_DIR):
    os.makedirs(base, exist_ok=True)
print('local scratch:', LOCAL_RUN_DIR)
print('Drive run dir:', RUN_DRIVE_DIR)
!df -h /content | tail -1


## 3. Clone/checkout the repo (refuses to overwrite uncommitted work)

In [ ]:
# 3. Clone if absent; otherwise fetch + checkout at COMMIT (or origin/BRANCH).
import subprocess, sys
os.chdir('/content')
if not os.path.exists(REPO):
    !git clone $REPO_URL $REPO
os.chdir(REPO)
dirty = subprocess.run(['git','status','--porcelain'], capture_output=True, text=True).stdout.strip()
if dirty:
    raise RuntimeError('repo has uncommitted changes -- refusing to checkout over them:\n' + dirty)
!git fetch -q origin
ref = COMMIT if COMMIT else f'origin/{BRANCH}'
!git checkout -q $ref
!git log -1 --oneline
for req in ('crl/offline_audit.py','crl/rockfall_ant.py','crl/train.py',
            'scripts/naive_rockfall_v2_crl.py','scripts/verify_offline_d4rl.py',
            'scripts/naive_rockfall_v2_crl.py',
            'scripts/diagnose_naive_rockfall.py'):
    if not os.path.exists(req):
        raise RuntimeError(f'{req} missing -- push main from the workstation and rerun.')
print('rockfall pipeline files present -- checkout OK')

## 4. Dependencies (preserve Colab's preinstalled GPU JAX -- do not alter)

In [ ]:
# 4. Install deps WITHOUT disturbing Colab's GPU JAX (pin jax/jaxlib/numpy).
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['MUJOCO_GL'] = 'egl'
import jax, jaxlib, numpy
hold = [f'jax=={jax.__version__}', f'jaxlib=={jaxlib.__version__}', f'numpy=={numpy.__version__}']
def pip(*a): subprocess.run([sys.executable,'-m','pip','install','-q',*a], check=True)
print('Colab JAX', jax.__version__, '| devices:', jax.devices())
pip('--no-deps', 'dm-haiku', 'optax', 'chex')
pip('jmp', 'tabulate', 'toolz', 'etils', 'tensorboardX', 'mujoco', 'imageio', 'imageio-ffmpeg', *hold)
print('post-install JAX', jax.__version__, '| devices:', jax.devices())

## 5. GPU / environment verification

In [ ]:
# 5. Require an accelerator; record env meta to Drive.
import hashlib, json, platform, mujoco
os.chdir('/content/'+REPO)
commit = subprocess.run(['git','rev-parse','HEAD'], capture_output=True, text=True).stdout.strip()
meta = {'run_id': RUN_ID, 'python': platform.python_version(), 'jax': jax.__version__,
        'backend': jax.default_backend(), 'devices':[str(d) for d in jax.devices()],
        'mujoco': mujoco.__version__, 'git_commit': commit, 'env': ENV_NAME}
if REQUIRE_GPU and jax.default_backend() == 'cpu':
    raise RuntimeError('no accelerator -- Runtime > Change runtime type > GPU')
!nvidia-smi -L || true
json.dump(meta, open(f'{RUN_DRIVE_DIR}/meta_env.json','w'), indent=2)
print(json.dumps(meta, indent=1))

## 6. Dataset: verify the repo-shipped npz (sha256)

The 300-episode pilot npz is committed at the pinned commit -- nothing to upload.


In [ ]:
# 6. Verify the repo-shipped rockfall npz; fail fast on any mismatch.
def sha256(path, chunk=1<<20):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
if not os.path.exists(LOCAL_DATASET_PATH):
    raise FileNotFoundError('dataset missing from the clone: ' + LOCAL_DATASET_PATH
                            + ' (is COMMIT pinned to 4dd0f8f or later?)')
got = sha256(LOCAL_DATASET_PATH)
if got != DATASET_SHA256:
    raise SystemExit('dataset sha mismatch: got ' + got + ' exp ' + DATASET_SHA256)
print('dataset OK:', LOCAL_DATASET_PATH)
print('sha256:', got[:16], '...')


## 7. Pre-training offline audit -- must PASS before any training

In [ ]:
# 7. Static offline gates (G1-G8) on the REAL rockfall dataset.
os.chdir('/content/'+REPO); sys.path.insert(0, '/content/'+REPO)
from crl import offline_audit
from crl.config import Config
from crl import envs as envs_mod
_c = Config(env_name=ENV_NAME, offline_dataset=LOCAL_DATASET_PATH)
envs_mod.make_env(ENV_NAME, _c, seed=0)   # fills obs/goal/action dims
passed, gates, rep = offline_audit.run_static_audit(LOCAL_DATASET_PATH, _c)
print('offline_audit gates:', {g:('PASS' if ok else 'FAIL') for g,ok in gates.items()})
fp = rep['fingerprint']
print(f"  sha256={fp['sha256'][:16]}  eps={fp['n_episodes']}  trans={fp['n_transitions']}  obs={fp['obs_shape']}")
assert passed, 'offline_audit FAILED -- see gates'

## 8. (resume) restore latest.pkl from Drive

In [ ]:
# 8. If RESUME, pull the last checkpoint from Drive into local scratch.
import glob, shutil, os
if RESUME:
    for f in glob.glob(f'{RUN_DRIVE_DIR}/*.pkl') + [f'{RUN_DRIVE_DIR}/metrics.json',
                                                    f'{RUN_DRIVE_DIR}/offline_dataset.sha256']:
        if os.path.exists(f): shutil.copy2(f, f'{LOCAL_RUN_DIR}/{os.path.basename(f)}')
    print('restored:', sorted(os.path.basename(p) for p in glob.glob(f'{LOCAL_RUN_DIR}/*.pkl')))
else:
    print('fresh run (RESUME=False)')

## 9. TensorBoard

In [ ]:
tb = f'{LOCAL_RUN_DIR}/tb'
%load_ext tensorboard
%tensorboard --logdir $tb

## 10. Launch training (live stream)

Trains to Drive-mirrored local scratch. Eval every 10k in the rockfall env.

In [ ]:
# 10. Launch naive offline CRL on the rockfall dataset (LIVE streaming).
os.chdir('/content/'+REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
cmd = [sys.executable, '-u', 'scripts/naive_rockfall_v2_crl.py',
       '--npz', LOCAL_DATASET_PATH, '--steps', str(MAX_UPDATES),
       '--seed', str(SEED), '--ckpt-dir', LOCAL_RUN_DIR,
              '--p-active', str(P_ACTIVE),
              '--horizon', str(HORIZON)]
if RESUME: cmd.append('--resume')
print(' '.join(cmd)); print('-'*70)
proc = subprocess.Popen(cmd, env={**os.environ}, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()
print(f'\n[driver exited {rc}]')

## 11. Mirror checkpoints to Drive

In [ ]:
# 11. Copy checkpoints + metrics to the persistent Drive dir.
import glob, shutil, os, json
n = 0
for f in glob.glob(f'{LOCAL_RUN_DIR}/*.pkl') + glob.glob(f'{LOCAL_RUN_DIR}/*.json') + glob.glob(f'{LOCAL_RUN_DIR}/*.sha256'):
    shutil.copy2(f, f'{RUN_DRIVE_DIR}/{os.path.basename(f)}'); n += 1
print(f'mirrored {n} files to {RUN_DRIVE_DIR}')
mp = f'{LOCAL_RUN_DIR}/metrics.json'
if os.path.exists(mp):
    for e in json.load(open(mp))[-8:]:
        print({k: e.get(k) for k in ('step','success','min_dist','final_dist')})

## 12. Done -- download for the workstation diagnosis

Training + checkpoints are mirrored to Drive. Behavioural characterisation
(route distribution, trigger-avoidance scan, paired mask-flip) runs on the
WORKSTATION: download `best.pkl` (+ `metrics.json`) from the Drive run dir into
`artifacts/naive_rockfall_crl/` and run
`python scripts/diagnose_naive_rockfall.py --ckpt artifacts/naive_rockfall_crl/best.pkl`.


## 12. Authoritative evaluation at H=800 (final + best) and packaging
Reconcile harness scores naive + teacher + center + blind at H=800; the naive behavioural diagnosis adds route/exposure/drop/leakage/gaming. These are the authoritative results (training-eval was monitoring only).

In [ ]:
# 12a. Authoritative reconcile eval (all policies) at H=800 -- final + best
import subprocess, sys, os
os.chdir('/content/'+REPO)
RES = f'{RUN_DRIVE_DIR}/eval_h800'
os.makedirs(RES, exist_ok=True)
for tag in ('final','best'):
    ck = f'{LOCAL_RUN_DIR}/{tag}.pkl'
    print('=== reconcile', tag, '===', flush=True)
    r = subprocess.run([sys.executable,'scripts/reconcile_rockfall_eval.py',
        '--naive-ckpt', ck, '--p-active','0.30','--horizon','800',
        '--k','100','--n-nat','200',
        '--out', f'{RES}/reconcile_{tag}.json'], capture_output=True, text=True)
    print(r.stdout[-1500:]); print(r.stderr[-600:] if r.returncode else '')


In [ ]:
# 12b. Naive behavioural diagnosis at H=800 (route/exposure/drop/leakage/gaming)
for tag in ('final','best'):
    ck = f'{LOCAL_RUN_DIR}/{tag}.pkl'
    print('=== diagnose', tag, '===', flush=True)
    r = subprocess.run([sys.executable,'scripts/diagnose_naive_rockfall.py',
        '--v2','--p-active','0.30','--horizon','800','--ckpt', ck,
        '--out-dir', f'{RES}/diag_{tag}'], capture_output=True, text=True)
    print(r.stdout[-1200:]); print(r.stderr[-600:] if r.returncode else '')


In [ ]:
# 12c. Package final artifacts for download (checkpoints + metrics + eval)
import shutil
pkg = f'/content/{RUN_ID}_bundle'
os.makedirs(pkg, exist_ok=True)
for f in glob.glob(f'{LOCAL_RUN_DIR}/*.pkl')+glob.glob(f'{LOCAL_RUN_DIR}/*.json')+glob.glob(f'{LOCAL_RUN_DIR}/*.sha256'):
    shutil.copy2(f, pkg)
shutil.copytree(RES, f'{pkg}/eval_h800', dirs_exist_ok=True)
zp = shutil.make_archive(f'{RUN_DRIVE_DIR}/{RUN_ID}_bundle','zip', pkg)
print('packaged ->', zp)
print('Download from Drive:', f'{RUN_DRIVE_DIR}/{RUN_ID}_bundle.zip')
